# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [1]:
import sys
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

print(sys.executable)


import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    # Mémoire réduite : machine à 8 Go de RAM, éviter de saturer le système
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    # Active un vrai traceback Python si l'UDF crash, au lieu d'une erreur Java opaque
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

/Users/nina/big-data-spark/venv/bin/python3.11


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/13 11:44:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.0


## 1 - Chemins & constantes

In [2]:
TRAIN_PATH   = "./data/Train_5/"
TEST_PATH    = "./data/Test_5/"
OUTPUT_PREDS = "./output/predictions/"
MODEL_PATH   = "./output/model/"
TARGET_SIZE  = (64, 64)

In [3]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

In [4]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        raw = img.tobytes()
        n = len(raw)
        pixels = list(struct.unpack(f"{n}B", raw))
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
    )
    return (
        raw
        .select(
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            F.col("content").alias("raw_bytes"),
        )
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show()

Images train : 10
Images test  : 10
+----------+-------+--------------------+
|  image_id|  label|              pixels|
+----------+-------+--------------------+
|000139.jpg|tulipes|[180.0, 129.0, 79...|
|000137.jpg|tulipes|[184.0, 155.0, 50...|
|000063.jpg|    lys|[154.0, 2.0, 1.0,...|
|000140.jpg|tulipes|[118.0, 102.0, 84...|
|000061.jpg|    lys|[131.0, 119.0, 82...|
|000138.jpg|tulipes|[81.0, 15.0, 54.0...|
|000062.jpg|    lys|[205.0, 198.0, 16...|
|000064.jpg|    lys|[118.0, 145.0, 10...|
|000141.jpg|tulipes|[0.0, 72.0, 0.0, ...|
|000065.jpg|    lys|[164.0, 190.0, 21...|
+----------+-------+--------------------+



## 3 - Prétraitement 

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit,
et on ajoute une colonne `pixels_gray` : niveaux de gris normalisés en [0.0, 1.0].

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

`pixels` est une liste aplatie `[R,G,B, R,G,B, ...]` de taille 64*64*3 = 12288.
L'UDF regroupe les valeurs par 3, applique la formule et renvoie les pixels en niveaux de gris

In [5]:
def rgb_to_grayscale(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    return gray

gray_udf = F.udf(rgb_to_grayscale, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_grayscale", gray_udf(F.col("pixels")))

train_preprocessed_gray_df = preprocess(train_parsed_df)
test_preprocessed_gray_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_gray_df.select("image_id", "label", "pixels_grayscale").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_gray_df
    .select(F.size(F.col("pixels_grayscale")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_grayscale (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                        pixels_grayscale|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[30.616, 32.018, 33.491, 35.263, 37.9...|
|000004.jpg|tulipes|[93.374, 96.787, 99.684, 102.755, 104...|
|000004.jpg|    lys|[207.335, 205.221, 205.107, 205.107, ...|
|000002.jpg|    lys|[135.315, 136.201, 135.087, 136.087, ...|
|000001.jpg|    lys|[22.399, 18.274, 60.137, 102.701, 108...|
+----------+-------+----------------------------------------+
only showing top 5 rows

Taille pixels_grayscale (attendu 4096) : 4096


In [6]:
def rgb_to_normalized_gray(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        # normalise la valeur résultante dans l'intervalle [0, 1] au lieu de [0, 255],
        gray.append(g_value / 255.0)
    return gray

gray_udf = F.udf(rgb_to_normalized_gray, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_gray", gray_udf(F.col("pixels")))

train_preprocessed_norm_gray_df = preprocess(train_parsed_df)
test_preprocessed_norm_gray_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_norm_gray_df.select("image_id", "label", "pixels_gray").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_norm_gray_df
    .select(F.size(F.col("pixels_gray")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_gray (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                             pixels_gray|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[0.120062746, 0.12556079, 0.13133726,...|
|000004.jpg|tulipes|[0.36617255, 0.37955686, 0.39091766, ...|
|000004.jpg|    lys|[0.8130784, 0.80478823, 0.8043412, 0....|
|000002.jpg|    lys|[0.53064704, 0.5341216, 0.52975297, 0...|
|000001.jpg|    lys|[0.087839216, 0.07166275, 0.23583138,...|
+----------+-------+----------------------------------------+
only showing top 5 rows

Taille pixels_gray (attendu 4096) : 4096


## ML couleurs - prétraitement en bytes

In [7]:
# Prétraitement couleur (bytes) : RGB brut 0-255, encodé en BinaryType
from pyspark.sql.types import BinaryType

def pixels_to_bytes(pixels):
    if pixels is None:
        return None
    int_pixels = [int(p) for p in pixels]
    return struct.pack(f"{len(int_pixels)}B", *int_pixels)

bytes_udf = F.udf(pixels_to_bytes, BinaryType())

def preprocess_color_bytes(df):
    return df.withColumn("pixels_color_bytes", bytes_udf(F.col("pixels")))

train_color_bytes_df = preprocess_color_bytes(train_parsed_df)
test_color_bytes_df  = preprocess_color_bytes(test_parsed_df)

print("Aperçu après prétraitement :")
train_color_bytes_df.select("image_id", "label", "pixels_color_bytes").show(5, truncate=40)

expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_color_bytes_df
    .select(F.length(F.col("pixels_color_bytes")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_color_bytes en octets (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                      pixels_color_bytes|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[1A 24 0F 1A 26 11 1B 28 11 1D 2A 11 ...|
|000004.jpg|tulipes|[65 5F 41 69 62 45 6E 64 47 72 67 48 ...|
|000004.jpg|    lys|[C8 D1 DA C6 CF D7 C6 CF D6 C6 CF D6 ...|
|000002.jpg|    lys|[8E 83 8C 8F 84 8C 8E 83 8A 8F 84 8B ...|
|000001.jpg|    lys|[14 1B 05 0E 18 00 3A 3F 33 66 67 67 ...|
+----------+-------+----------------------------------------+
only showing top 5 rows

Taille pixels_color_bytes en octets (attendu 12288) : 12288


In [8]:
# ML - Random Forest sur les features couleur (bytes)
# collect() autorisé uniquement en section ML, pour repasser en numpy/sklearn

def to_numpy(df, feature_col, label_col="label", is_binary=False):
    rows = df.select(feature_col, label_col).collect()  # autorisé ici
    if is_binary:
        X = np.array([
            list(struct.unpack(f"{len(r[feature_col])}B", r[feature_col]))
            for r in rows
        ], dtype=np.float32)
    else:
        X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = np.array([r[label_col] for r in rows])
    return X, y

X_train, y_train = to_numpy(train_color_bytes_df, "pixels_color_bytes", is_binary=True)
X_test,  y_test  = to_numpy(test_color_bytes_df,  "pixels_color_bytes", is_binary=True)

print(f"X_train : {X_train.shape}, X_test : {X_test.shape}")

param_dist = {
    "n_estimators": randint(50, 200),
    "max_depth": randint(5, 30),
    "min_samples_split": randint(2, 10),
}

rf = RandomForestClassifier(random_state=42)
search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=10, cv=3, random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)

best_rf = search.best_estimator_
print("Meilleurs hyperparamètres :", search.best_params_)

y_pred = best_rf.predict(X_test)

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")
print(f"F1        : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")
print()
print(classification_report(y_test, y_pred, zero_division=0))

X_train : (10, 12288), X_test : (10, 12288)
Meilleurs hyperparamètres : {'max_depth': 28, 'min_samples_split': 5, 'n_estimators': 87}
Accuracy  : 0.400
Precision : 0.400
Recall    : 0.400
F1        : 0.400

              precision    recall  f1-score   support

         lys       0.40      0.40      0.40         5
     tulipes       0.40      0.40      0.40         5

    accuracy                           0.40        10
   macro avg       0.40      0.40      0.40        10
weighted avg       0.40      0.40      0.40        10



## ML couleurs normalisées


## ML grayscale

Py4JJavaError: An error occurred while calling o400.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 8 in stage 35.0 failed 1 times, most recent failure: Lost task 8.0 in stage 35.0 (TID 114) (Alia executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:685)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:128)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:106)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:596)
	at org.apache.spark.InterruptibleIterator.hasNext(Int

## ML grayscale normalisé